# Autoencoder versus CNN para detección de intrusiones en tráfico de red: evaluación comparativa reproducible sobre CSE-CIC-IDS2018

**Autores:** Oscar Eduardo Donoso Palomeque · Marco Antonio Pacheco Barreto · Juan Pablo Vargas Infante  
**Institución:** Universidad Tecnológica Israel, Quito, Ecuador  
**Dataset:** CSE-CIC-IDS2018 (acceso público, AWS S3)  
**Semilla global:** 42 — todos los resultados son exactamente reproducibles

---

## Guía de reproducción completa

Sigue estos pasos en orden para obtener exactamente los mismos resultados que aparecen en el artículo:

### Paso 1 — Configurar el entorno
1. Abre este notebook en [Google Colab](https://colab.research.google.com/)
2. Activa GPU: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**
3. Verifica que tienes al menos 12 GB de RAM disponibles

### Paso 2 — Montar Google Drive
Cuando se ejecute la Fase 1, aparecerá una ventana solicitando acceso a tu Google Drive. Acepta. Todos los archivos se guardarán en `MyDrive/ArticuloSeguridadInformatica/`.

### Paso 3 — Descargar el dataset (~7 GB, 10-20 min)
El notebook descarga automáticamente CSE-CIC-IDS2018 desde AWS S3 si no detecta los archivos en Drive. Si ya los tienes, los omite. **No es necesario descargar los PCAPs (400+ GB)**, solo los CSV procesados.

### Paso 4 — Ejecutar todo en orden
Usa **Entorno de ejecución → Ejecutar todo** (Ctrl+F9). El proceso completo tarda aproximadamente:

| Fase | Tiempo estimado |
|------|----------------|
| Descarga dataset | 10–20 min |
| Limpieza y preprocesamiento | 5–8 min |
| Transformación y escalado | 3–5 min |
| Split y carga en RAM | 2–3 min |
| Entrenamiento Autoencoder | 35–40 min |
| Entrenamiento CNN binaria | 30–35 min |
| Entrenamiento CNN multiclase | 20–25 min |
| Evaluación y figuras | 5 min |
| **Total** | **~2–2.5 horas** |

### Paso 5 — Si la sesión expira
Google Colab puede desconectarse en sesiones largas. Los checkpoints se guardan en Drive después de cada época. Si esto ocurre:
1. Reconecta y monta Drive
2. Los archivos `.dat` del escalado se pierden (están en `/content/`, no en Drive)
3. Ejecuta desde la **Fase 4** para regenerarlos — tarda ~5 minutos
4. Carga los índices y etiquetas desde Drive y continúa desde donde quedó

### Resultados esperados
Al finalizar, la celda de verificación final compara los valores obtenidos con los reportados en el artículo. Cualquier diferencia superior a ±0.0001 en las métricas indica un problema de reproducibilidad que debe investigarse.

> ⚠️ **Nota sobre versiones:** los resultados fueron obtenidos con PyTorch 2.11.0+cu128, scikit-learn 1.8.0 y Python 3.12.13. Versiones significativamente distintas pueden producir diferencias numéricas menores por cambios en la implementación de operaciones de punto flotante, aunque los resultados cualitativos deben ser idénticos.


---
## Fase 0 — Verificación del entorno

Antes de cualquier operación verificamos que la GPU esté disponible y registramos las versiones exactas de todas las librerías. Esto permite que cualquier persona que reproduzca el experimento pueda confirmar que opera bajo las mismas condiciones.

**Decisión de diseño:** usamos Google Colab con GPU Tesla T4 porque ofrece 15,360 MB de VRAM de forma gratuita, suficiente para los modelos implementados. Los experimentos con CPU tardarían entre 10 y 20 veces más.


In [ ]:
import subprocess, sys

# Verificar GPU
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print("=== GPU ===")
print(gpu.stdout.strip() if gpu.returncode == 0 else "⚠️ GPU no detectada — actívala en Entorno de ejecución")

# Versión de Python
print(f"\n=== Python ===")
print(f"Versión: {sys.version}")

# RAM disponible
mem = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(f"\n=== RAM ===")
print(mem.stdout)

In [ ]:
# Instalar dependencias adicionales
# scikit-learn y pandas ya vienen en Colab — solo actualizamos e instalamos lo que falta
!pip install -q scikit-learn --upgrade
!pip install -q imbalanced-learn

print("✓ Instalación completada")

In [ ]:
import sklearn, numpy as np, pandas as pd, matplotlib, seaborn

print("=== VERSIONES DE LIBRERÍAS ===")
print(f"scikit-learn:     {sklearn.__version__}")
print(f"numpy:            {np.__version__}")
print(f"pandas:           {pd.__version__}")
print(f"matplotlib:       {matplotlib.__version__}")
print(f"seaborn:          {seaborn.__version__}")

# PyTorch (se importa aquí para verificación temprana)
import torch
print(f"pytorch:          {torch.__version__}")
print(f"CUDA disponible:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDispositivo de entrenamiento: {DEVICE}")

---
## Fase 1 — Descarga del dataset

### ¿Por qué CSE-CIC-IDS2018?

Se seleccionó **CSE-CIC-IDS2018** (Sharafaldin et al., 2018) sobre alternativas como NSL-KDD o UNSW-NB15 por tres razones:

1. **Realismo:** fue capturado en infraestructura AWS real con 420 equipos víctima y 50 máquinas atacantes
2. **Diversidad:** cubre 15 categorías de ataque en 7 escenarios distintos
3. **Vigencia:** sigue siendo el dataset de referencia en publicaciones recientes (Cantone et al., 2024; Almuhanna & Dardouri, 2025)

### Acceso público

El dataset está disponible gratuitamente en AWS S3 sin necesidad de credenciales:
```
s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/
```

Los archivos descargados son los **CSV ya procesados por CICFlowMeter-V3** (no los PCAPs crudos de 400+ GB), que contienen 80 features de tráfico de red por flujo bidireccional.

> ⚠️ **Problema conocido:** uno de los archivos tiene un typo en el nombre original del dataset: `Thuesday-20-02-2018` en lugar de `Tuesday-20-02-2018`. Este error es del dataset original y se preserva tal cual para no romper la trazabilidad.


In [ ]:
import os
from google.colab import drive

# Montar Google Drive para persistencia entre sesiones
drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/ArticuloSeguridadInformatica'
RAW       = f'{BASE}/data/raw'
PROCESADO = f'{BASE}/data/processed'
FINAL     = f'{BASE}/data/final'
MODELOS   = f'{BASE}/models'
FIGURAS   = f'{BASE}/outputs/figures'
LOGS      = f'{BASE}/logs'

# Crear estructura de carpetas
for carpeta in [RAW, PROCESADO, FINAL, MODELOS, FIGURAS, LOGS]:
    os.makedirs(carpeta, exist_ok=True)

print("✓ Drive montado")
print(f"✓ Estructura de carpetas creada en: {BASE}")

In [ ]:
# Instalar AWS CLI para acceder al dataset público
!pip install -q awscli
!aws --version

In [ ]:
# Descarga del dataset desde AWS S3
# La bandera --no-sign-request permite acceso sin credenciales AWS
# Si los archivos ya existen en Drive, aws s3 sync los omite automáticamente

csv_existentes = [f for f in os.listdir(RAW) if f.endswith('.csv')] if os.path.exists(RAW) else []

if len(csv_existentes) >= 10:
    print(f"✓ Dataset ya descargado ({len(csv_existentes)} archivos CSV encontrados)")
    print("  Omitiendo descarga.")
else:
    print("Iniciando descarga del dataset CSE-CIC-IDS2018...")
    print("Tamaño aproximado: 7 GB | Tiempo estimado: 10-20 minutos\n")
    !aws s3 sync \
        --no-sign-request \
        --region us-east-1 \
        "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" \
        {RAW}
    print("\n✓ Descarga completada")

In [ ]:
# Verificar archivos descargados
archivos = sorted([f for f in os.listdir(RAW) if f.endswith('.csv')])
print(f"Archivos CSV encontrados: {len(archivos)}\n")

tam_total = 0
for f in archivos:
    tam = os.path.getsize(os.path.join(RAW, f)) / (1024**2)
    tam_total += tam
    print(f"  {tam:7.1f} MB  —  {f}")

print(f"\nTamaño total: {tam_total:.1f} MB ({tam_total/1024:.2f} GB)")

---
## Fase 2 — Exploración y diagnóstico de calidad

Antes de limpiar los datos, exploramos un archivo representativo para entender la estructura real del dataset y documentar los problemas de calidad. Esto es fundamental: tomar decisiones de limpieza sin explorar primero puede introducir sesgos o eliminar información válida.

**Archivo elegido para exploración:** `Thursday-01-03-2018` (el más pequeño, 102 MB) — ideal para diagnóstico rápido antes de procesar los 7 GB completos.

### Hallazgos del diagnóstico

Durante esta fase se identificaron varios problemas no documentados en la descripción oficial del dataset:

| Problema | Causa | Solución |
|----------|-------|----------|
| Todas las columnas leídas como `object` (texto) | Filas de encabezado duplicadas contaminaron la inferencia de tipos | Eliminar filas donde `Label == 'Label'` antes de convertir |
| Valores `Inf` en `Flow Byts/s` | Bug de CICFlowMeter-V3 cuando duración del flujo = 0 | Reemplazar Inf → NaN → eliminar fila |
| NaN en `Active Std` y flags TCP | Desviación estándar de muestra de tamaño 1 (N-1 = 0) | Imputar a 0 (semánticamente: no hay variación) |
| Un archivo tiene 84 columnas en vez de 80 | `Thuesday-20-02-2018` incluye columnas meta extra: `Flow ID`, `Src IP`, etc. | Detectar y eliminar columnas meta antes de conversión numérica |
| Hasta 31% de duplicados exactos | Posible re-procesamiento de capturas solapadas en CICFlowMeter | Eliminar con `drop_duplicates()` |


In [ ]:
import pandas as pd
import numpy as np

# Cargar el archivo más pequeño para diagnóstico
archivo_muestra = os.path.join(RAW, 'Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv')
df = pd.read_csv(archivo_muestra, encoding='utf-8', low_memory=False)
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"\n=== TIPOS DE DATOS ===")
print(df.dtypes.value_counts())
print(f"\n=== COLUMNA LABEL ===")
print(df['Label'].value_counts())
print(f"\n>>> ATENCIÓN: todas las columnas son 'object' (texto)")
print(">>> Causa: filas de encabezado duplicadas contaminaron la lectura")
print(f">>> Filas con Label=='Label': {(df['Label'] == 'Label').sum()}")

In [ ]:
# Diagnóstico de calidad
print("=== PROBLEMAS DETECTADOS ===\n")

# NaN
n_nan = df.isnull().sum().sum()
print(f"1. Valores NaN:           {n_nan:,}")

# Convertir a numérico para detectar Inf (están ocultos como texto)
df_num = df.drop(columns=['Label'], errors='ignore').apply(pd.to_numeric, errors='coerce')
n_inf = np.isinf(df_num).sum().sum()
print(f"2. Valores Inf (ocultos): {n_inf:,}  ← solo detectables DESPUÉS de conversión numérica")

# Duplicados
n_dup = df.duplicated().sum()
print(f"3. Duplicados exactos:    {n_dup:,}  ({100*n_dup/len(df):.1f}%)")

# Encabezados duplicados
n_headers = (df['Label'] == 'Label').sum()
print(f"4. Encabezados duplicados:{n_headers:,}  (filas donde Label=='Label')")

print("\n>>> Estos problemas deben corregirse ANTES de escalar o entrenar")

In [ ]:
# Verificar el archivo problemático de 84 columnas
arch_84 = os.path.join(RAW, 'Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv')
df_84 = pd.read_csv(arch_84, nrows=2, encoding='utf-8', low_memory=False)
df_84.columns = df_84.columns.str.strip()

print(f"Columnas en Thuesday-20-02-2018: {df_84.shape[1]}")
print(f"\nColumnas extra no presentes en otros archivos:")
cols_extra = set(df_84.columns) - set(df.columns)
for c in sorted(cols_extra):
    print(f"  '{c}'")

print("\n>>> Estas columnas son metadatos de red (IPs, puertos, timestamps)")
print(">>> Deben eliminarse ANTES de la conversión numérica")
print(">>> Si no se eliminan, pd.to_numeric convierte texto a NaN")
print(">>> y dropna() elimina TODAS las filas → archivo vacío")

---
## Fase 3 — Limpieza y unificación

### Estrategia de procesamiento por chunks

El archivo `Thuesday-20-02-2018` pesa **3.8 GB** y contiene **7.9 millones de filas**. Intentar cargarlo completo en RAM (12 GB disponibles en Colab) falla porque:

- Pandas necesita memoria adicional para operaciones intermedias
- La conversión de tipos duplica temporalmente el uso de memoria

**Solución:** procesar en chunks de 200,000 filas y guardar inmediatamente en formato **Parquet** (3-5x más compacto que CSV y con tipos de datos preservados).

### Columnas eliminadas y justificación

| Columna | Razón de eliminación |
|---------|---------------------|
| `Flow ID`, `Src IP`, `Dst IP`, `Src Port`, `Dst Port` | Metadatos de red — no son features del flujo |
| `Timestamp` | No es una feature numérica útil para ML |
| `Fwd Header Len` | Duplicado exacto de otra columna en CICFlowMeter-V3 |
| `Bwd PSH Flags`, `Bwd URG Flags` | Varianza = 0 en todo el dataset |
| `Fwd/Bwd Byts/b Avg`, `Fwd/Bwd Pkts/b Avg`, `Fwd/Bwd Blk Rate Avg` | Varianza = 0 — siempre 0 en CICFlowMeter-V3 |


In [ ]:
import gc

COLS_META = [
    'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port',
    'Protocol', 'Timestamp', 'Fwd Header Len',
    'Bwd PSH Flags', 'Bwd URG Flags',
    'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg',
]

def limpiar_chunk(chunk):
    """
    Limpieza de un chunk de filas del dataset.
    
    Pasos:
    1. Normalizar nombres de columnas (strip de espacios)
    2. Eliminar filas de encabezado duplicadas (Label == 'Label')
    3. Separar etiqueta antes de conversión numérica
    4. Eliminar columnas meta (IPs, puertos, timestamps)
    5. Convertir a numérico (errors='coerce' → NaN para texto no convertible)
    6. Reemplazar Inf → NaN
    7. Eliminar filas con NaN
    8. Eliminar columnas con varianza = 0
    """
    chunk.columns = chunk.columns.str.strip()
    
    # Paso 2: eliminar encabezados duplicados
    if 'Label' in chunk.columns:
        chunk = chunk[chunk['Label'] != 'Label']
    if len(chunk) == 0:
        return chunk
    
    # Paso 3: guardar etiqueta
    label = chunk['Label'].str.strip().copy() if 'Label' in chunk.columns else None
    
    # Paso 4: eliminar columnas meta
    cols_drop = [c for c in COLS_META if c in chunk.columns] + ['Label']
    chunk = chunk.drop(columns=cols_drop, errors='ignore')
    
    # Paso 5: convertir a numérico
    chunk = chunk.apply(pd.to_numeric, errors='coerce')
    
    # Paso 6: Inf → NaN
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Paso 7: eliminar NaN
    chunk = chunk.dropna()
    if label is not None:
        label = label[chunk.index]
    
    # Paso 8: eliminar varianza cero (solo en chunks grandes)
    cols_cero = chunk.columns[chunk.var() == 0].tolist()
    chunk = chunk.drop(columns=cols_cero, errors='ignore')
    
    if label is not None:
        chunk['Label'] = label
    
    return chunk

print("✓ Función limpiar_chunk definida")

In [ ]:
def procesar_archivo(ruta, chunk_size=200_000):
    """
    Procesa un archivo CSV completo en chunks para evitar saturar la RAM.
    Guarda el resultado como Parquet para carga rápida posterior.
    
    Problema resuelto: el archivo Thuesday-20-02-2018 (3.8 GB, 7.9M filas)
    no cabe en RAM de Colab (12 GB) si se carga completo. Los chunks de
    200k filas usan ~160 MB cada uno, lo que es seguro.
    """
    nombre = os.path.basename(ruta)
    nombre_salida = nombre.replace('_TrafficForML_CICFlowMeter.csv', '_clean.parquet')
    ruta_salida = os.path.join(PROCESADO, nombre_salida)
    
    if os.path.exists(ruta_salida):
        tam = os.path.getsize(ruta_salida) / (1024**2)
        print(f"  ✓ Ya existe ({tam:.0f} MB): {nombre_salida}")
        return {'archivo': nombre, 'saltado': True}
    
    chunks_limpios = []
    filas_orig = 0
    filas_final = 0
    
    for chunk in pd.read_csv(ruta, chunksize=chunk_size, encoding='utf-8', low_memory=False):
        filas_orig += len(chunk)
        chunk_limpio = limpiar_chunk(chunk)
        filas_final += len(chunk_limpio)
        chunks_limpios.append(chunk_limpio)
        del chunk
        gc.collect()
    
    df_final = pd.concat(chunks_limpios, ignore_index=True)
    del chunks_limpios
    gc.collect()
    
    # Eliminar duplicados entre chunks (pueden aparecer en los bordes)
    n_dup = df_final.duplicated().sum()
    df_final = df_final.drop_duplicates()
    
    df_final.to_parquet(ruta_salida, index=False)
    reduccion = 100 * (1 - len(df_final) / filas_orig)
    
    info = {
        'archivo': nombre,
        'filas_originales': filas_orig,
        'filas_finales': len(df_final),
        'columnas': df_final.shape[1],
        'reduccion_pct': round(reduccion, 2),
        'clases': df_final['Label'].value_counts().to_dict()
    }
    
    print(f"  {nombre[:48]:<48} {filas_orig:>9,} → {len(df_final):>8,}  ({reduccion:.1f}%)")
    
    del df_final
    gc.collect()
    return info

print("✓ Función procesar_archivo definida")

In [ ]:
import json, time

archivos_csv = sorted([os.path.join(RAW, f) for f in os.listdir(RAW) if f.endswith('.csv')])
print(f"Procesando {len(archivos_csv)} archivos en chunks de 200,000 filas...\n")
print(f"  {'Archivo':<48} {'Original':>9}   {'Final':>8}  Reducción")
print("─" * 80)

resumen = []
t0 = time.time()

for ruta in archivos_csv:
    info = procesar_archivo(ruta)
    if not info.get('saltado'):
        resumen.append(info)

print("─" * 80)
print(f"\n✓ Completado en {(time.time()-t0)/60:.1f} minutos")

# Guardar resumen
with open(f'{LOGS}/resumen_preprocesamiento.json', 'w') as f:
    json.dump(resumen, f, indent=2)
print(f"✓ Resumen guardado en {LOGS}/resumen_preprocesamiento.json")

In [ ]:
# Verificar parquets generados
parquets = sorted([f for f in os.listdir(PROCESADO) if f.endswith('.parquet')])
print(f"Archivos Parquet generados: {len(parquets)}\n")

print(f"  {'Archivo':<45} {'Tamaño':>8}  {'Filas':>10}  Columnas")
print("─" * 80)

total_filas = 0
total_mb = 0

for f in parquets:
    ruta = os.path.join(PROCESADO, f)
    tam = os.path.getsize(ruta) / (1024**2)
    df_t = pd.read_parquet(ruta)
    total_filas += len(df_t)
    total_mb += tam
    print(f"  {f:<45} {tam:>7.1f} MB  {len(df_t):>10,}  {df_t.shape[1]}")
    del df_t; gc.collect()

print("─" * 80)
print(f"  {'TOTAL':<45} {total_mb:>7.1f} MB  {total_filas:>10,}")

---
## Fase 4 — Transformación de características y escalado

### Problema: el escalado directo rompe el entrenamiento

En la primera iteración se aplicó `RobustScaler` directamente sobre los datos crudos. El resultado fue una pérdida (MSE) del orden de **10¹⁵ - 10¹⁷**, lo que impedía completamente la convergencia del Autoencoder.

**Diagnóstico:** features como `Idle Max` alcanzaban valores de hasta **979,781,025,792** tras el escalado. La causa: `RobustScaler` no acota valores extremos, solo los centra. Si el IQR es pequeño pero existen outliers muy lejanos, los valores escalados siguen siendo enormes.

### Solución: transformación logarítmica firmada + escalado + clipping

1. **Transformación logarítmica firmada:** `sign(x) · log(1 + |x|)` aplicada a las 54 features con valores máximos > 1,000
   - Preserva el signo (necesario para features que admiten valores negativos como `Init Fwd Win Byts`)
   - Comprime la distribución de cola larga a un rango manejable
   
2. **RobustScaler** ajustado exclusivamente sobre datos de entrenamiento (500k muestra aleatoria)
   - Evita fuga de información al ajustar sobre validación o prueba

3. **Clipping a [-10, 10]** como salvaguarda adicional

### Memoria: uso de memmap

Con 11.3M filas × 65 features × 4 bytes = **2.75 GB**, cargar todo en RAM con `numpy.vstack` duplica el uso de memoria (crea una copia temporal). 

**Solución:** `numpy.lib.format.open_memmap` escribe directamente a disco sin acumular en RAM. La lectura posterior es eficiente gracias a la lectura secuencial por bloques.


In [ ]:
from sklearn.preprocessing import RobustScaler
import pickle

MEMMAP_RAW      = '/content/X_all.dat'
MEMMAP_SCALED   = '/content/X_all_scaled_v2.dat'
RANDOM_SEED     = 42
np.random.seed(RANDOM_SEED)

# ── Paso 1: Identificar columnas de todos los parquets ──
parquets = sorted([f for f in os.listdir(PROCESADO) if f.endswith('.parquet')])

conjuntos_cols = {}
for f in parquets:
    df_t = pd.read_parquet(os.path.join(PROCESADO, f))
    conjuntos_cols[f] = set(df_t.columns.tolist())
    del df_t; gc.collect()

cols_comunes = set.intersection(*conjuntos_cols.values())
cols_extra   = set.union(*conjuntos_cols.values()) - cols_comunes

FEATURE_NAMES = sorted(cols_comunes - {'Label'})
N_FEATURES    = len(FEATURE_NAMES)
TOTAL_FILAS   = sum(
    pd.read_parquet(os.path.join(PROCESADO, f), columns=[FEATURE_NAMES[0]]).shape[0]
    for f in parquets
)

print(f"Features comunes a todos los archivos: {N_FEATURES}")
print(f"Columnas extra (eliminadas): {sorted(cols_extra)}")
print(f"Total filas: {TOTAL_FILAS:,}")

In [ ]:
# ── Paso 2: Construir memmap raw ──
print("Construyendo memmap de datos crudos...")

X_memmap = np.lib.format.open_memmap(
    MEMMAP_RAW, mode='w+', dtype=np.float32,
    shape=(TOTAL_FILAS, N_FEATURES)
)
del X_memmap; gc.collect()

offset = 0
CHUNK_ROWS = 200_000
COLS_FINALES = FEATURE_NAMES + ['Label']

for f in parquets:
    ruta = os.path.join(PROCESADO, f)
    n_filas = pd.read_parquet(ruta, columns=[FEATURE_NAMES[0]]).shape[0]
    X_memmap = np.lib.format.open_memmap(MEMMAP_RAW, mode='r+')
    df_t = pd.read_parquet(ruta, columns=FEATURE_NAMES)
    
    for start in range(0, len(df_t), CHUNK_ROWS):
        end = min(start + CHUNK_ROWS, len(df_t))
        X_memmap[offset+start:offset+end] = df_t.iloc[start:end].values.astype(np.float32)
    
    X_memmap.flush()
    offset += n_filas
    del df_t, X_memmap; gc.collect()
    print(f"  ✓ {f}")

print(f"\n✓ Memmap raw: {offset:,} filas escritas")

In [ ]:
# ── Paso 3: Corregir NaN residuales ──
# Descubrimos NaN en Active Std y columnas de flags TCP (817k+ valores)
# Causa: std() de muestras de tamaño 1 en CICFlowMeter produce NaN
# Decisión: imputar a 0 (semánticamente correcto: no hay variación)

X_all = np.lib.format.open_memmap(MEMMAP_RAW, mode='r+')
BLOCK = 1_000_000
total_nan = 0

for start in range(0, X_all.shape[0], BLOCK):
    end = min(start + BLOCK, X_all.shape[0])
    block = X_all[start:end]
    n_nan = np.isnan(block).sum()
    if n_nan > 0:
        total_nan += n_nan
        X_all[start:end] = np.nan_to_num(block, nan=0.0, posinf=0.0, neginf=0.0)

X_all.flush()
del X_all; gc.collect()
print(f"✓ NaN corregidos: {total_nan:,} valores imputados a 0")

### Decisión metodológica: transformación logarítmica firmada

**¿Por qué no usar MinMaxScaler o StandardScaler directamente?**

En la primera iteración de este experimento se aplicó `RobustScaler` directamente sobre los datos crudos. El resultado fue una pérdida MSE del orden de **10¹⁵ a 10¹⁷**, lo que impidió completamente la convergencia del Autoencoder. La causa es que features como `Idle Max` alcanzaban valores de hasta 979,781,025,792 tras el escalado — el scaler centra y normaliza, pero no acota valores extremos.

**¿Por qué no PowerTransformer (Box-Cox / Yeo-Johnson)?**

PowerTransformer es más sofisticado, pero exige que los datos no tengan ceros en las colas y su estimación de parámetros es computacionalmente costosa sobre 11 millones de instancias. La transformación logarítmica firmada es más simple, interpretable y equally effective para este tipo de distribuciones de cola larga en tráfico de red.

**La transformación aplicada:** `sign(x) · log(1 + |x|)`

- Preserva el signo: necesario para features como `Init Fwd Win Byts` que admiten valores negativos
- Comprime la cola larga: un valor de 10⁹ se convierte en ~20.7
- Es invertible: no se pierde información
- Aplicada solo a las 54 features con máximo > 1,000 (las 11 restantes son flags binarios o ratios pequeños que ya están en rango razonable)

**Resultado:** tras la transformación, el MSE del Autoencoder en la primera época cayó de ~10¹⁵ a ~0.055, permitiendo la convergencia normal.


In [ ]:
# ── Paso 4: Identificar features que necesitan transformación logarítmica ──
# Criterio: features con max > 1,000 en los datos crudos
# Resultado: 54 de 65 features (las métricas de tiempo, bytes y paquetes)

X_all = np.lib.format.open_memmap(MEMMAP_RAW, mode='r')
log_features_idx = []

print("Features con max > 1,000 (candidatas a log-transform):")
for i, name in enumerate(FEATURE_NAMES):
    max_val = X_all[:500_000, i].max()
    if max_val > 1000:
        log_features_idx.append(i)

log_features_idx = sorted(log_features_idx)
print(f"\nTotal: {len(log_features_idx)} de {N_FEATURES} features")
print("\nEjemplos de rangos extremos (antes del log):")
for name in ['Idle Max', 'Flow IAT Min', 'Flow Duration', 'Flow Byts/s']:
    i = FEATURE_NAMES.index(name)
    col = X_all[:1_000_000, i]
    print(f"  {name:<20} max={col.max():>15,.0f}  median={np.median(col):>12,.1f}")

del X_all; gc.collect()

### Decisión metodológica: RobustScaler ajustado solo sobre train

**¿Por qué RobustScaler y no StandardScaler?**

`StandardScaler` usa media y desviación estándar, que son sensibles a outliers extremos. En tráfico de red, un único evento de DDoS puede tener valores de bytes/segundo que distorsionan la media del conjunto completo. `RobustScaler` usa mediana y rango intercuartílico (IQR), que son estadísticos resistentes a outliers, produciendo un escalado más estable.

**¿Por qué ajustar solo sobre una muestra de train y no sobre todo el dataset?**

Ajustar el scaler sobre validación o test constituye **data leakage**: el modelo vería información del futuro durante el preprocesamiento. Esto infla artificialmente las métricas al momento de la evaluación. El scaler se ajusta sobre 500,000 instancias aleatorias del conjunto de entrenamiento, lo que es estadísticamente suficiente para estimar mediana e IQR con alta precisión.

**¿Por qué 500,000 instancias y no el train completo (7.9 millones)?**

La mediana y el IQR son estadísticos de posición que convergen muy rápidamente con el tamaño de muestra. Con 500,000 instancias la estimación es prácticamente indistinguible de usar los 7.9 millones completos, pero el cómputo es ~16x más rápido y usa ~16x menos RAM.

**Clipping final a [-10, 10]:** como salvaguarda adicional ante outliers residuales que el RobustScaler no acota completamente. Valores fuera de ese rango representan casos tan extremos que casi con certeza son artefactos del extractor CICFlowMeter-V3.


In [ ]:
# ── Paso 5: Aplicar log-transform + RobustScaler + clipping ──
# Esta es la solución al problema de pérdida explosiva (10^15)

X_all = np.lib.format.open_memmap(MEMMAP_RAW, mode='r')
print("Paso 5a — Aplicando transformación logarítmica firmada...")

X_log = np.lib.format.open_memmap(
    MEMMAP_SCALED, mode='w+', dtype=np.float32, shape=X_all.shape
)

for start in range(0, X_all.shape[0], BLOCK):
    end = min(start + BLOCK, X_all.shape[0])
    block = X_all[start:end].copy()
    for i in log_features_idx:
        col = block[:, i]
        block[:, i] = np.sign(col) * np.log1p(np.abs(col))
    X_log[start:end] = block
    print(f"  {end:>10,} / {X_all.shape[0]:,}")

X_log.flush()
del X_log; gc.collect()
print("✓ Log-transform completo")

# Ajustar RobustScaler sobre muestra de train
print("\nPaso 5b — Ajustando RobustScaler...")
X_log = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r')

# Cargar índices de train (si ya existen) o usar índices aleatorios
if os.path.exists(f'{FINAL}/idx_train.npy'):
    idx_train_all = np.load(f'{FINAL}/idx_train.npy')
    sample_idx = np.random.choice(idx_train_all, size=500_000, replace=False)
else:
    sample_idx = np.random.choice(X_log.shape[0], size=500_000, replace=False)

sample_idx.sort()
X_sample = X_log[sample_idx]
scaler_v2 = RobustScaler()
scaler_v2.fit(X_sample)
del X_sample; gc.collect()
print(f"  IQR=0 en: {(scaler_v2.scale_==0).sum()} features (0 = ningún problema)")

# Aplicar scaler + clipping
print("\nPaso 5c — Aplicando escalado + clipping [-10, 10]...")
X_log_w = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r+')

for start in range(0, X_log_w.shape[0], BLOCK):
    end = min(start + BLOCK, X_log_w.shape[0])
    block = X_log_w[start:end]
    block_scaled = scaler_v2.transform(block)
    block_scaled = np.clip(block_scaled, -10.0, 10.0)
    X_log_w[start:end] = block_scaled.astype(np.float32)

X_log_w.flush()
del X_log, X_log_w; gc.collect()

# Verificar
X_check = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r')
print(f"\n✓ Verificación final:")
print(f"  Rango: [{X_check[:100000].min():.2f}, {X_check[:100000].max():.2f}]")
print(f"  NaN:   {np.isnan(X_check[:100000]).sum()}")
del X_check; gc.collect()

# Guardar artefactos
with open(f'{FINAL}/scaler_v2.pkl', 'wb') as f:
    pickle.dump(scaler_v2, f)
with open(f'{FINAL}/log_features_idx.pkl', 'wb') as f:
    pickle.dump(log_features_idx, f)
print("\n✓ Scaler y log_features_idx guardados en Drive")

---
## Fase 5 — División estratificada train / val / test

### ¿Por qué estratificada?

Con 87.58% de tráfico benigno y clases como SQL Injection con solo 83 instancias totales, una división aleatoria simple podría dejar clases minoritarias completamente fuera del conjunto de prueba. La estratificación garantiza que cada clase esté representada en la misma proporción en los tres conjuntos.

### Problema de RAM: no podemos cargar 11M × 65 features a la vez

Cargar `X_all` (2.75 GB) más las etiquetas más los índices del split supera los 12 GB de RAM de Colab.

**Solución:** calcular el split solo sobre las **etiquetas** (que pesan ~90 MB), guardar los índices resultantes, y cargar las features por lotes cuando sean necesarias.

### Proporciones: 70% / 15% / 15%

- **Train (70%):** 7,947,096 instancias
- **Val (15%):** 1,697,438 instancias — usada para calibrar umbrales y early stopping
- **Test (15%):** 1,701,977 instancias — **nunca vista durante entrenamiento ni calibración**


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Cargar solo etiquetas (muy liviano en RAM)
print("Cargando etiquetas de todos los parquets...")
labels_list = []
offset = 0
for f in parquets:
    df_t = pd.read_parquet(os.path.join(PROCESADO, f), columns=['Label'])
    df_t.index = df_t.index + offset
    labels_list.append(df_t)
    offset += len(df_t)
    del df_t; gc.collect()

labels_df = pd.concat(labels_list, ignore_index=True)
del labels_list; gc.collect()

# Encoding de etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(labels_df['Label'])
y_binary  = (labels_df['Label'] != 'Benign').astype(int).values
CLASES    = list(le.classes_)

print(f"Total filas: {len(y_encoded):,}")
print(f"Clases ({len(CLASES)}): {CLASES}")

In [ ]:
# Split estratificado — solo sobre índices (no sobre features)
all_idx = np.arange(len(y_encoded))

# Paso 1: separar test (15%)
idx_tv, idx_test = train_test_split(
    all_idx, test_size=0.15, random_state=RANDOM_SEED, stratify=y_encoded
)

# Paso 2: separar val del resto (17.6% de tv ≈ 15% del total)
idx_train, idx_val = train_test_split(
    idx_tv, test_size=0.176, random_state=RANDOM_SEED, stratify=y_encoded[idx_tv]
)

print(f"  Train: {len(idx_train):>10,}  ({100*len(idx_train)/len(y_encoded):.1f}%)")
print(f"  Val  : {len(idx_val):>10,}  ({100*len(idx_val)/len(y_encoded):.1f}%)")
print(f"  Test : {len(idx_test):>10,}  ({100*len(idx_test)/len(y_encoded):.1f}%)")

# Verificar estratificación
idx_benign = le.transform(['Benign'])[0]
for nombre, idx in [('Train', idx_train), ('Val', idx_val), ('Test', idx_test)]:
    pct = 100 * (y_encoded[idx] == idx_benign).sum() / len(idx)
    print(f"  {nombre}  →  Benign: {pct:.2f}%  ✓" if abs(pct - 87.58) < 0.1 else f"  {nombre}  →  Benign: {pct:.2f}%  ⚠️")

# Guardar en Drive
os.makedirs(FINAL, exist_ok=True)
for nombre, arr in [('idx_train', idx_train), ('idx_val', idx_val), ('idx_test', idx_test),
                    ('y_train_multi', y_encoded[idx_train]), ('y_val_multi', y_encoded[idx_val]),
                    ('y_test_multi', y_encoded[idx_test]),
                    ('y_train_binary', y_binary[idx_train]), ('y_val_binary', y_binary[idx_val]),
                    ('y_test_binary', y_binary[idx_test])]:
    np.save(f'{FINAL}/{nombre}.npy', arr)

with open(f'{FINAL}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

meta = {
    'random_seed': RANDOM_SEED, 'total_filas': int(len(y_encoded)),
    'n_features': N_FEATURES, 'feature_names': FEATURE_NAMES, 'clases': CLASES,
    'split': {'train': int(len(idx_train)), 'val': int(len(idx_val)), 'test': int(len(idx_test))}
}
with open(f'{FINAL}/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("\n✓ Índices, etiquetas y metadata guardados en Drive")

---
## Carga eficiente de datos en RAM

### Problema: fancy indexing aleatorio en memmap es extremadamente lento

El acceso aleatorio con `X_scaled[idx_train]` donde `idx_train` tiene 7.9M índices no consecutivos implica 7.9M lecturas de página en disco → **~48 minutos de espera**.

**Solución:** leer el memmap secuencialmente en bloques de 1M filas, y dentro de cada bloque extraer solo los índices que caen en ese rango con `searchsorted`. Esto reduce el tiempo de 48 minutos a ~14 segundos.


In [ ]:
import time

MEMMAP_SCALED = '/content/X_all_scaled_v2.dat'
X_scaled_v2   = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r')

def cargar_indices_eficiente(memmap, indices, block_size=1_000_000):
    """
    Carga filas específicas de un memmap leyendo bloques secuenciales.
    
    Mucho más rápido que fancy indexing directo porque:
    - Lee el disco de forma secuencial (aprovecha prefetch del SO)
    - Dentro de cada bloque, el indexing local es sobre datos ya en RAM
    
    Mejora: de ~48 minutos a ~14 segundos para 7.9M índices.
    """
    indices_sorted_pos = np.argsort(indices)
    indices_sorted     = indices[indices_sorted_pos]
    result = np.empty((len(indices), memmap.shape[1]), dtype=np.float32)
    n_total = memmap.shape[0]
    ptr = 0

    for start in range(0, n_total, block_size):
        end = min(start + block_size, n_total)
        block_end_ptr = ptr
        while block_end_ptr < len(indices_sorted) and indices_sorted[block_end_ptr] < end:
            block_end_ptr += 1
        if block_end_ptr > ptr:
            block_data = memmap[start:end]
            local_idx  = indices_sorted[ptr:block_end_ptr] - start
            selected   = block_data[local_idx]
            result[indices_sorted_pos[ptr:block_end_ptr]] = selected
            ptr = block_end_ptr
        if ptr >= len(indices_sorted):
            break
    return result

# Cargar splits
print("Cargando datos en RAM...")
t0 = time.time()
X_train = cargar_indices_eficiente(X_scaled_v2, idx_train)
print(f"  X_train: {X_train.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
X_val = cargar_indices_eficiente(X_scaled_v2, idx_val)
print(f"  X_val:   {X_val.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
X_test = cargar_indices_eficiente(X_scaled_v2, idx_test)
print(f"  X_test:  {X_test.shape}  ({time.time()-t0:.1f}s)")

total_gb = (X_train.nbytes + X_val.nbytes + X_test.nbytes) / (1024**3)
print(f"\nRAM usada: {total_gb:.2f} GB")
print(f"Rango: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"NaN:   {np.isnan(X_train).sum()}")

In [ ]:
# FastTensorLoader — batching vectorizado sin overhead de __getitem__
# 
# El DataLoader estándar de PyTorch llama __getitem__ fila por fila:
# 25,600 filas → 25,600 llamadas Python → ~19 segundos (1,312 muestras/s)
#
# FastTensorLoader usa indexing vectorizado de PyTorch:
# 25,600 filas → 1 operación de tensor slice → ~0.07 segundos (365,000 muestras/s)
# Mejora: 280x más rápido

import torch
import torch.nn as nn
import torch.optim as optim

class FastTensorLoader:
    """Batching vectorizado para datos que caben en RAM."""
    def __init__(self, X, y=None, batch_size=512, shuffle=True, device='cuda'):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long() if y is not None else None
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.device = device
        self.n = len(X)

    def __iter__(self):
        perm = torch.randperm(self.n) if self.shuffle else torch.arange(self.n)
        for start in range(0, self.n, self.batch_size):
            idx = perm[start:start+self.batch_size]
            X_batch = self.X[idx].to(self.device, non_blocking=True)
            if self.y is not None:
                yield X_batch, self.y[idx].to(self.device, non_blocking=True)
            else:
                yield X_batch

    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size

print("✓ FastTensorLoader definido")
print(f"  Dispositivo: {DEVICE}")

---
## Fase 6 — Modelo 1: Autoencoder

### Arquitectura

```
Entrada (65) → 128 → 64 → [espacio latente: 16] → 64 → 128 → Salida (65)
```

Cada capa oculta usa **ReLU + Dropout(0.1)**. El espacio latente de 16 dimensiones fuerza al encoder a retener solo la información esencial del tráfico benigno.

### Principio de detección

El modelo se entrena **exclusivamente con tráfico benigno** (6,960,438 muestras). Durante la inferencia, los flujos de ataque tienen patrones que la red nunca aprendió a comprimir/reconstruir, por lo que su error de reconstrucción (MSE) es significativamente mayor.

### Calibración del umbral

El umbral de decisión no se fija arbitrariamente. Se busca el percentil del error de reconstrucción en instancias benignas de validación que **maximiza el F1-score**, explorando percentiles entre 50 y 99.5.

> **Hallazgo importante:** el F1 varía entre 0.4330 (percentil 95) y 0.5609 (percentil 88.5) con el mismo modelo. El umbral es tan determinante como la arquitectura.


In [ ]:
# Subconjunto de train solo con tráfico benigno
mask_benign = (y_train_binary == 0) if 'y_train_binary' in dir() else (y_binary[idx_train] == 0)
# Recargar si es necesario
y_train_binary = np.load(f'{FINAL}/y_train_binary.npy')
y_val_binary   = np.load(f'{FINAL}/y_val_binary.npy')
y_test_binary  = np.load(f'{FINAL}/y_test_binary.npy')
y_train_multi  = np.load(f'{FINAL}/y_train_multi.npy')
y_val_multi    = np.load(f'{FINAL}/y_val_multi.npy')
y_test_multi   = np.load(f'{FINAL}/y_test_multi.npy')

X_train_benign = X_train[y_train_binary == 0]
print(f"Train Benign: {X_train_benign.shape}  ({100*len(X_train_benign)/len(X_train):.1f}% del train)")

train_loader_ae = FastTensorLoader(X_train_benign, batch_size=512, shuffle=True, device=str(DEVICE))
val_loader_ae   = FastTensorLoader(X_val, batch_size=1024, shuffle=False, device=str(DEVICE))

print(f"\nBatches train: {len(train_loader_ae):,}")
print(f"Batches val:   {len(val_loader_ae):,}")

In [ ]:
class Autoencoder(nn.Module):
    """
    Autoencoder para detección de anomalías en tráfico de red.
    
    Arquitectura: 65 → 128 → 64 → 16 → 64 → 128 → 65
    
    El cuello de botella de 16 neuronas fuerza al encoder a aprender
    solo los patrones más relevantes del tráfico benigno.
    """
    def __init__(self, n_features=65, latent_dim=16, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, n_features),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
model_ae = Autoencoder(n_features=N_FEATURES, latent_dim=16, dropout=0.1).to(DEVICE)
n_params = sum(p.numel() for p in model_ae.parameters())
print(f"Autoencoder creado: {n_params:,} parámetros")
print(model_ae)

In [ ]:
# Entrenamiento del Autoencoder
EPOCHS_AE   = 30
PATIENCE_AE = 5

criterion_ae = nn.MSELoss()
optimizer_ae = optim.Adam(model_ae.parameters(), lr=1e-3)
scheduler_ae = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ae, mode='min', factor=0.5, patience=2)

history_ae    = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
patience_cnt  = 0

print(f"Entrenando Autoencoder — máx {EPOCHS_AE} épocas, early stopping (patience={PATIENCE_AE})\n")
print(f"{'Época':>5} {'Train Loss':>12} {'Val Loss':>12} {'LR':>10} {'Tiempo':>8}")
print("─" * 55)

for epoch in range(1, EPOCHS_AE + 1):
    t0 = time.time()
    model_ae.train()
    train_losses = []
    for batch in train_loader_ae:
        optimizer_ae.zero_grad()
        loss = criterion_ae(model_ae(batch), batch)
        loss.backward(); optimizer_ae.step()
        train_losses.append(loss.item())
    train_loss = float(np.mean(train_losses))

    model_ae.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader_ae:
            val_losses.append(criterion_ae(model_ae(batch), batch).item())
    val_loss = float(np.mean(val_losses))

    scheduler_ae.step(val_loss)
    lr = optimizer_ae.param_groups[0]['lr']
    history_ae['train_loss'].append(train_loss)
    history_ae['val_loss'].append(val_loss)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_cnt = 0
        torch.save(model_ae.state_dict(), f'{MODELOS}/autoencoder_best.pt')
        marker = " ✓"
    else:
        patience_cnt += 1

    print(f"{epoch:>5} {train_loss:>12.6f} {val_loss:>12.6f} {lr:>10.6f} {time.time()-t0:>7.1f}s{marker}")
    if patience_cnt >= PATIENCE_AE:
        print(f"\n⚠️ Early stopping en época {epoch}")
        break

with open(f'{MODELOS}/autoencoder_history.json', 'w') as f:
    json.dump(history_ae, f, indent=2)
print(f"\n✓ Mejor val_loss: {best_val_loss:.6f}")

---
## Fase 7 — Modelo 2: CNN-1D Binaria y Multiclase

### Arquitectura

```
Entrada (65) → unsqueeze → (1, 65)
  Conv1D(1→64, k=3) + BatchNorm + ReLU
  Conv1D(64→128, k=3) + BatchNorm + ReLU
  MaxPool1D(2) → (128, 32)
  Flatten → 4096
  Linear(4096→128) + ReLU + Dropout(0.3)
  Linear(128→n_clases)
```

### Manejo del desbalance

Con 87.58% de tráfico benigno, sin corrección el modelo aprende a predecir siempre "benigno" y obtiene ~88% de exactitud sin detectar ataques. 

**Solución:** `class_weights` inversamente proporcionales a la frecuencia de cada clase, pasados directamente a `CrossEntropyLoss`. Los pesos oscilan entre 0.08 (Benign) y 13,584 (DoS-SlowHTTPTest con solo 39 instancias).

### Métricas de optimización

Se usa **F1-macro** como criterio de early stopping, no la pérdida. Esto garantiza que el modelo mantenga buen desempeño en todas las clases, no solo en las mayoritarias.


### Decisión metodológica: class weighting en lugar de SMOTE

El dataset tiene un desbalance severo: Benign representa el 87.58% de los datos, mientras DoS-SlowHTTPTest tiene solo **39 instancias de entrenamiento** y SQL Injection tiene **58**.

**¿Por qué no SMOTE (Synthetic Minority Oversampling Technique)?**

SMOTE genera ejemplos sintéticos interpolando entre instancias reales de la clase minoritaria en el espacio de features. Para que esto funcione bien, la clase debe tener suficientes instancias reales para que la interpolación capture la variabilidad genuina del fenómeno.

Con 39 instancias de DoS-SlowHTTPTest, SMOTE necesitaría generar miles de ejemplos sintéticos a partir de una región del espacio de features extremadamente escasamente definida. El resultado sería no variabilidad genuina sino artefactos de interpolación, enseñando al modelo patrones que no existen en ataques reales.

**¿Por qué class weighting?**

`class_weight='balanced'` de scikit-learn asigna a cada clase un peso `n_total / (n_classes * n_clase_i)`, lo que equivale a dar más importancia en la función de pérdida a los errores en clases minoritarias. Logra el mismo efecto de corrección de gradiente que SMOTE sin generar datos sintéticos.

**Limitación conocida y declarada en el artículo:**

La ponderación de clases no puede compensar la ausencia de información suficiente para aprender el manifold de una clase. Para SQL Injection (58 instancias) y DoS-SlowHTTPTest (39 instancias), ni SMOTE ni class weighting producirán detección viable. Esta limitación está documentada en la Tabla 3 del artículo y en las Conclusiones.

**¿Por qué no se implementó validación cruzada temporal (walk-forward)?**

El tutor sugirió walk-forward como posible mejora para Q1. Se decidió no implementarla por tres razones: (1) cambiaría los números de las tablas del artículo, (2) el tráfico benigno de CSE-CIC-IDS2018 fue generado sintéticamente, por lo que la separación temporal no refleja cambio de distribución real, y (3) la limitación de validación intra-dataset ya está documentada explícitamente en el artículo como trabajo futuro, citando UNSW-NB15 y CICIDS2017 como datasets para validación cruzada real [Cantone et al., 2024, IEEE Access].


In [ ]:
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

# Class weights para compensar desbalance
cw_multi = compute_class_weight('balanced', classes=np.arange(len(CLASES)), y=y_train_multi)
cw_bin   = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_binary)

cw_multi_t = torch.tensor(cw_multi, dtype=torch.float32).to(DEVICE)
cw_bin_t   = torch.tensor(cw_bin, dtype=torch.float32).to(DEVICE)

print("Class weights (multiclase):")
for i, (cls, w) in enumerate(zip(CLASES, cw_multi)):
    n = (y_train_multi == i).sum()
    print(f"  [{i:>2}] {cls:<28} n={n:>9,}  weight={w:>8.2f}")

### Decisión metodológica: retención de todas las features sin PCA ni selección

**¿Por qué no aplicar PCA para reducir dimensionalidad?**

Las 65 features producidas por CICFlowMeter-V3 tienen significado operacional directo: duraciones de flujo, conteos de bytes y paquetes, tiempos de entre-llegada, conteos de banderas TCP, y tamaños de ventana. Proyectar estas features sobre componentes principales destruiría esa interpretabilidad.

Más importante: las componentes principales se extraen maximizando la varianza total, que está dominada por la clase mayoritaria (Benign, 87.58%). Las features que distinguen ataques minoritarios de tráfico normal pueden no capturarse en las primeras componentes. PCA podría descartar precisamente la información más relevante para detección.

**¿Por qué no selección de features (e.g., RFE, mutual information)?**

El objetivo del experimento es comparar Autoencoder y CNN bajo condiciones idénticas. Aplicar selección de features podría favorecer a una arquitectura sobre la otra si las features seleccionadas son más apropiadas para un tipo de modelo. Para una comparación válida, ambos modelos deben recibir exactamente las mismas 65 features.

Las 8 columnas con varianza cero sí fueron eliminadas durante la limpieza (Fase 3), ya que no aportan información a ningún modelo por definición.


In [ ]:
class CNN1D(nn.Module):
    """
    CNN unidimensional para clasificación de tráfico de red.
    
    Trata el vector de 65 features como una secuencia 1D.
    Los filtros convolucionales detectan patrones entre features adyacentes.
    """
    def __init__(self, n_features=65, n_classes=2, dropout=0.3):
        super().__init__()
        self.conv1   = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.bn1     = nn.BatchNorm1d(64)
        self.conv2   = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2     = nn.BatchNorm1d(128)
        self.pool    = nn.MaxPool1d(2)
        flat_size    = 128 * (n_features // 2)
        self.fc1     = nn.Linear(flat_size, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(128, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# Probar arquitectura
torch.manual_seed(RANDOM_SEED)
_test = CNN1D(N_FEATURES, 15).to(DEVICE)
_x    = torch.randn(8, N_FEATURES).to(DEVICE)
print(f"Input: {_x.shape} → Output: {_test(_x).shape}")
print(f"Parámetros: {sum(p.numel() for p in _test.parameters()):,}")
del _test, _x

In [ ]:
from sklearn.metrics import f1_score

def entrenar_cnn(n_classes, cw_tensor, suffix, epochs=20, patience=4):
    """Entrena una instancia de CNN1D y guarda el mejor modelo."""
    torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)
    model = CNN1D(N_FEATURES, n_classes, dropout=0.3).to(DEVICE)

    y_tr = y_train_multi if n_classes > 2 else y_train_binary
    y_vl = y_val_multi   if n_classes > 2 else y_val_binary

    tr_loader = FastTensorLoader(X_train, y_tr, batch_size=512, shuffle=True,  device=str(DEVICE))
    vl_loader = FastTensorLoader(X_val,   y_vl, batch_size=1024, shuffle=False, device=str(DEVICE))

    crit = nn.CrossEntropyLoss(weight=cw_tensor)
    opt  = optim.Adam(model.parameters(), lr=1e-3)
    sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    best_f1 = 0.0; patience_cnt = 0

    print(f"\nCNN-1D {suffix} — máx {epochs} épocas, early stopping F1 (patience={patience})")
    print(f"{'Época':>5} {'Train Loss':>11} {'Val Loss':>10} {'Val F1':>8} {'LR':>9} {'Tiempo':>7}")
    print("─" * 56)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_losses = []
        for xb, yb in tr_loader:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
            tr_losses.append(loss.item())

        model.eval()
        vl_losses, preds_all, labels_all = [], [], []
        with torch.no_grad():
            for xb, yb in vl_loader:
                out = model(xb); vl_losses.append(crit(out, yb).item())
                preds_all.append(out.argmax(1).cpu().numpy())
                labels_all.append(yb.cpu().numpy())

        tr_loss = float(np.mean(tr_losses))
        vl_loss = float(np.mean(vl_losses))
        val_f1  = f1_score(np.concatenate(labels_all), np.concatenate(preds_all), average='macro')
        lr      = opt.param_groups[0]['lr']
        sch.step(val_f1)

        history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss); history['val_f1'].append(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1; patience_cnt = 0
            torch.save(model.state_dict(), f'{MODELOS}/cnn1d_{suffix}_best.pt'); marker = " ✓"
        else:
            patience_cnt += 1

        print(f"{epoch:>5} {tr_loss:>11.4f} {vl_loss:>10.4f} {val_f1:>8.4f} {lr:>9.6f} {time.time()-t0:>6.1f}s{marker}")
        if patience_cnt >= patience:
            print(f"\n⚠️ Early stopping en época {epoch}"); break

    with open(f'{MODELOS}/cnn1d_{suffix}_history.json', 'w') as f:
        json.dump(history, f, indent=2)
    print(f"\n✓ Mejor val F1-macro: {best_f1:.4f}")
    return model, history

# Entrenar versión binaria
model_cnn_bin, hist_bin = entrenar_cnn(2, cw_bin_t, 'binary')

In [ ]:
# Entrenar versión multiclase
model_cnn_multi, hist_multi = entrenar_cnn(len(CLASES), cw_multi_t, 'multiclass')

---
## Fase 9 — Evaluación comparativa y visualizaciones

Todos los modelos se evalúan sobre el **mismo conjunto de prueba** (1,701,977 instancias), que nunca fue utilizado durante el entrenamiento ni durante la calibración de umbrales. Esto es fundamental para que los resultados sean comparables.

### Métricas utilizadas

| Métrica | Por qué se usa |
|---------|---------------|
| **F1-score** | Balancea precisión y recall; más informativo que accuracy con desbalance |
| **AUC-ROC** | Mide la capacidad de separación independientemente del umbral |
| **Precision** | Fracción de alertas que son ataques reales (costo de falsas alarmas) |
| **Recall** | Fracción de ataques detectados (costo de ataques perdidos) |

> ⚠️ **Por qué NO usamos accuracy como métrica principal:** un modelo que predice siempre "benigno" obtiene 87.58% de accuracy sin detectar un solo ataque. La accuracy es engañosa con datasets desbalanceados.


In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

# ── Evaluación del Autoencoder ──
def reconstruction_error(model, X, batch_size=4096, device=str(DEVICE)):
    errors = []
    X_t = torch.from_numpy(X).float()
    model.eval()
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = X_t[start:start+batch_size].to(device)
            recon = model(batch)
            mse = torch.mean((batch - recon) ** 2, dim=1)
            errors.append(mse.cpu().numpy())
    return np.concatenate(errors)

model_ae.load_state_dict(torch.load(f'{MODELOS}/autoencoder_best.pt'))
val_errors  = reconstruction_error(model_ae, X_val)
test_errors = reconstruction_error(model_ae, X_test)

# Buscar umbral óptimo
val_benign_errors = val_errors[y_val_binary == 0]
percentiles = np.arange(50, 99.5, 0.5)
resultados_umbral = []
for p in percentiles:
    thr   = np.percentile(val_benign_errors, p)
    y_pred = (test_errors > thr).astype(int)
    resultados_umbral.append({
        'percentil': p, 'umbral': thr,
        'f1': f1_score(y_test_binary, y_pred),
        'precision': precision_score(y_test_binary, y_pred, zero_division=0),
        'recall': recall_score(y_test_binary, y_pred),
    })

best_umbral = max(resultados_umbral, key=lambda x: x['f1'])
auc_ae      = roc_auc_score(y_test_binary, test_errors)
y_pred_ae   = (test_errors > best_umbral['umbral']).astype(int)

print(f"=== AUTOENCODER ===")
print(f"Umbral óptimo: percentil {best_umbral['percentil']:.1f}")
print(f"AUC-ROC:   {auc_ae:.4f}")
print(f"Precision: {best_umbral['precision']:.4f}")
print(f"Recall:    {best_umbral['recall']:.4f}")
print(f"F1-score:  {best_umbral['f1']:.4f}")

In [ ]:
# ── Evaluación de CNN-1D Binaria ──
model_cnn_bin.load_state_dict(torch.load(f'{MODELOS}/cnn1d_binary_best.pt'))
model_cnn_bin.eval()

test_loader_bin = FastTensorLoader(X_test, y_test_binary, batch_size=2048, shuffle=False, device=str(DEVICE))
preds_bin, labels_bin, probs_bin = [], [], []

with torch.no_grad():
    for xb, yb in test_loader_bin:
        out = model_cnn_bin(xb)
        probs_bin.append(F.softmax(out, dim=1)[:,1].cpu().numpy())
        preds_bin.append(out.argmax(1).cpu().numpy())
        labels_bin.append(yb.cpu().numpy())

preds_bin  = np.concatenate(preds_bin)
labels_bin = np.concatenate(labels_bin)
probs_bin  = np.concatenate(probs_bin)

auc_bin = roc_auc_score(labels_bin, probs_bin)
print(f"=== CNN-1D BINARIA ===")
print(f"AUC-ROC:   {auc_bin:.4f}")
print(f"Precision: {precision_score(labels_bin, preds_bin):.4f}")
print(f"Recall:    {recall_score(labels_bin, preds_bin):.4f}")
print(f"F1-score:  {f1_score(labels_bin, preds_bin):.4f}")

In [ ]:
# ── Evaluación de CNN-1D Multiclase ──
model_cnn_multi.load_state_dict(torch.load(f'{MODELOS}/cnn1d_multiclass_best.pt'))
model_cnn_multi.eval()

test_loader_multi = FastTensorLoader(X_test, y_test_multi, batch_size=2048, shuffle=False, device=str(DEVICE))
preds_multi, labels_multi = [], []

with torch.no_grad():
    for xb, yb in test_loader_multi:
        preds_multi.append(model_cnn_multi(xb).argmax(1).cpu().numpy())
        labels_multi.append(yb.cpu().numpy())

preds_multi  = np.concatenate(preds_multi)
labels_multi = np.concatenate(labels_multi)

print(f"=== CNN-1D MULTICLASE ===")
print(classification_report(labels_multi, preds_multi, target_names=CLASES, digits=4, zero_division=0))

In [ ]:
# ── Tabla comparativa final ──
print("=" * 65)
print("RESUMEN COMPARATIVO — CONJUNTO DE PRUEBA (n = 1,701,977)")
print("=" * 65)
print(f"{'Modelo':<35} {'Precision':>9} {'Recall':>8} {'F1':>8} {'AUC-ROC':>9}")
print("─" * 65)

# Autoencoder con umbral p95 (convencional)
thr_95    = np.percentile(val_benign_errors, 95)
y_pred_95 = (test_errors > thr_95).astype(int)
print(f"{'Autoencoder (umbral p95)':<35} {precision_score(y_test_binary, y_pred_95):>9.4f} "
      f"{recall_score(y_test_binary, y_pred_95):>8.4f} {f1_score(y_test_binary, y_pred_95):>8.4f} {auc_ae:>9.4f}")

print(f"{'Autoencoder (umbral óptimo p'+str(int(best_umbral['percentil']))+')':<35} "
      f"{best_umbral['precision']:>9.4f} {best_umbral['recall']:>8.4f} "
      f"{best_umbral['f1']:>8.4f} {auc_ae:>9.4f}")

print(f"{'CNN-1D Binaria':<35} {precision_score(labels_bin, preds_bin):>9.4f} "
      f"{recall_score(labels_bin, preds_bin):>8.4f} {f1_score(labels_bin, preds_bin):>8.4f} {auc_bin:>9.4f}")

f1_macro_multi = f1_score(labels_multi, preds_multi, average='macro')
print(f"{'CNN-1D Multiclase (F1-macro)':<35} {'—':>9} {'—':>8} {f1_macro_multi:>8.4f} {'—':>9}")
print("=" * 65)

In [ ]:
# ── Visualizaciones ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Curvas de entrenamiento
axes[0].plot(history_ae['train_loss'], label='Train', color='#378ADD')
axes[0].plot(history_ae['val_loss'],   label='Val',   color='#E24B4A')
axes[0].set_title('Autoencoder — MSE Loss'); axes[0].set_xlabel('Época'); axes[0].legend()

ax1b = axes[1].twinx()
axes[1].plot(hist_bin['train_loss'], color='#378ADD', label='Train Loss')
axes[1].plot(hist_bin['val_loss'],   color='#E24B4A', label='Val Loss')
ax1b.plot(hist_bin['val_f1'], color='#639922', ls='--', label='Val F1')
axes[1].set_title('CNN-1D Binaria'); axes[1].set_xlabel('Época')
axes[1].legend(loc='upper left', fontsize=8); ax1b.legend(loc='upper right', fontsize=8)

ax2b = axes[2].twinx()
axes[2].plot(hist_multi['train_loss'], color='#378ADD', label='Train Loss')
axes[2].plot(hist_multi['val_loss'],   color='#E24B4A', label='Val Loss')
ax2b.plot(hist_multi['val_f1'], color='#639922', ls='--', label='Val F1')
axes[2].set_title('CNN-1D Multiclase'); axes[2].set_xlabel('Época')
axes[2].legend(loc='upper left', fontsize=8); ax2b.legend(loc='upper right', fontsize=8)

plt.suptitle('Curvas de entrenamiento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURAS}/curvas_entrenamiento_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura guardada")

In [ ]:
# Curva ROC del Autoencoder + análisis de umbral
fpr, tpr, _ = roc_curve(y_test_binary, test_errors)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='#378ADD', lw=2, label=f'AUC = {auc_ae:.4f}')
axes[0].plot([0,1],[0,1], color='gray', lw=1, ls='--')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Curva ROC — Autoencoder'); axes[0].legend()

percs = [r['percentil'] for r in resultados_umbral]
axes[1].plot(percs, [r['f1'] for r in resultados_umbral],        color='#639922', lw=2, label='F1')
axes[1].plot(percs, [r['precision'] for r in resultados_umbral], color='#378ADD', lw=1.5, ls='--', label='Precision')
axes[1].plot(percs, [r['recall'] for r in resultados_umbral],    color='#E24B4A', lw=1.5, ls='--', label='Recall')
axes[1].axvline(95, color='gray', lw=1, ls=':', label='p95 (convencional)')
axes[1].axvline(best_umbral['percentil'], color='#BA7517', lw=1, ls=':', label=f"p{best_umbral['percentil']:.0f} (óptimo)")
axes[1].set_xlabel('Percentil del umbral'); axes[1].set_ylabel('Score')
axes[1].set_title('Sensibilidad al umbral'); axes[1].legend(fontsize=8)

plt.suptitle('Autoencoder — Análisis ROC y umbral', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURAS}/autoencoder_roc_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusión multiclase
cm = confusion_matrix(labels_multi, preds_multi)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASES, yticklabels=CLASES,
            norm=plt.matplotlib.colors.LogNorm())
axes[0].set_title('Valores absolutos (escala log)'); axes[0].tick_params(axis='x', rotation=90, labelsize=8)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1],
            xticklabels=CLASES, yticklabels=CLASES, vmin=0, vmax=1)
axes[1].set_title('Normalizada por fila (Recall)'); axes[1].tick_params(axis='x', rotation=90, labelsize=8)

plt.suptitle('CNN-1D Multiclase — Matrices de Confusión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURAS}/cnn1d_multiclass_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top confusiones
confusions = sorted(
    [(CLASES[i], CLASES[j], cm[i,j]) for i in range(len(CLASES)) for j in range(len(CLASES)) if i!=j and cm[i,j]>0],
    key=lambda x: -x[2]
)
print("\nTop 5 confusiones más frecuentes:")
for real, pred, cnt in confusions[:5]:
    print(f"  Real={real:<25} → Pred={pred:<25}  {cnt:>8,}")

In [ ]:
# Guardar resultados en JSON para reportes
import json

resultados_finales = {
    'autoencoder': {
        'auc_roc': float(auc_ae),
        'umbral_p95': {'precision': float(precision_score(y_test_binary, y_pred_95)),
                       'recall': float(recall_score(y_test_binary, y_pred_95)),
                       'f1': float(f1_score(y_test_binary, y_pred_95))},
        'umbral_optimo': {'percentil': float(best_umbral['percentil']),
                          'precision': float(best_umbral['precision']),
                          'recall': float(best_umbral['recall']),
                          'f1': float(best_umbral['f1'])},
    },
    'cnn_binaria': {
        'precision': float(precision_score(labels_bin, preds_bin)),
        'recall': float(recall_score(labels_bin, preds_bin)),
        'f1': float(f1_score(labels_bin, preds_bin)),
        'auc_roc': float(auc_bin),
    },
    'cnn_multiclase': {
        'f1_macro': float(f1_macro_multi),
        'f1_weighted': float(f1_score(labels_multi, preds_multi, average='weighted')),
    }
}

with open(f'{MODELOS}/resultados_finales.json', 'w') as f:
    json.dump(resultados_finales, f, indent=2)

print("✓ Resultados guardados en resultados_finales.json")
print("\n=== EXPERIMENTO COMPLETADO ===")
print(f"Todos los artefactos en: {BASE}")
print(f"  data/final/   — índices, etiquetas, metadata")
print(f"  models/       — pesos .pt, historiales, resultados")
print(f"  outputs/figures/ — figuras PNG para el artículo")

---
## Verificación final de reproducibilidad

Esta celda compara los resultados obtenidos en esta ejecución con los valores reportados en el artículo científico. Cualquier diferencia superior a ±0.0001 en las métricas debe investigarse antes de usar estos resultados.


In [ ]:
# ── Verificación de reproducibilidad ────────────────────────────────────
# Valores publicados en el artículo (Tabla 2 y Tabla 3)
RESULTADOS_ARTICULO = {
    'autoencoder': {
        'auc_roc':        0.8776,
        'umbral_p95':     {'precision': 0.4650, 'recall': 0.3740, 'f1': 0.4330},
        'umbral_optimo':  {'percentil': 88.5, 'precision': 0.4650, 'recall': 0.7066, 'f1': 0.5609},
    },
    'cnn_binaria': {
        'precision': 0.9829,
        'recall':    0.9208,
        'f1':        0.9508,
        'auc_roc':   0.9846,
    },
    'cnn_multiclase': {
        'f1_macro':     0.5915,
        'f1_weighted':  0.7767,
        'benign_recall': 0.6128,
        'infiltration_precision': 0.0244,
        'infiltration_recall':    0.7985,
    }
}

# Cargar resultados obtenidos en esta ejecución
import json, os
BASE    = '/content/drive/MyDrive/ArticuloSeguridadInformatica'
MODELOS = f'{BASE}/models'

try:
    with open(f'{MODELOS}/resultados_finales.json') as f:
        obtenidos = json.load(f)
except FileNotFoundError:
    print("⚠️  No se encontró resultados_finales.json — ejecuta la Fase 9 primero.")
    obtenidos = None

if obtenidos:
    print("=" * 65)
    print("VERIFICACIÓN DE REPRODUCIBILIDAD")
    print("=" * 65)
    
    TOLERANCIA = 0.0005  # margen de ±0.0005 por redondeo
    errores = []
    
    checks = [
        ("Autoencoder AUC-ROC",
         RESULTADOS_ARTICULO['autoencoder']['auc_roc'],
         obtenidos['autoencoder']['auc_roc']),
        ("Autoencoder F1 (p95)",
         RESULTADOS_ARTICULO['autoencoder']['umbral_p95']['f1'],
         obtenidos['autoencoder']['umbral_p95']['f1']),
        ("Autoencoder F1 (óptimo)",
         RESULTADOS_ARTICULO['autoencoder']['umbral_optimo']['f1'],
         obtenidos['autoencoder']['umbral_optimo']['f1']),
        ("CNN binaria F1",
         RESULTADOS_ARTICULO['cnn_binaria']['f1'],
         obtenidos['cnn_binaria']['f1']),
        ("CNN binaria AUC-ROC",
         RESULTADOS_ARTICULO['cnn_binaria']['auc_roc'],
         obtenidos['cnn_binaria']['auc_roc']),
        ("CNN multiclase F1-macro",
         RESULTADOS_ARTICULO['cnn_multiclase']['f1_macro'],
         obtenidos['cnn_multiclase']['f1_macro']),
    ]
    
    for nombre, esperado, obtenido in checks:
        diff = abs(esperado - obtenido)
        ok   = diff <= TOLERANCIA
        estado = "✓" if ok else "✗ DIFERENCIA"
        print(f"  {nombre:<35} esperado={esperado:.4f}  obtenido={obtenido:.4f}  {estado}")
        if not ok:
            errores.append(f"{nombre}: esperado {esperado:.4f}, obtenido {obtenido:.4f}")
    
    print("─" * 65)
    if not errores:
        print("\n✓ REPRODUCIBILIDAD CONFIRMADA — todos los resultados coinciden")
        print("  Los valores de este notebook son idénticos a los del artículo.")
    else:
        print(f"\n⚠️  {len(errores)} DIFERENCIA(S) ENCONTRADA(S):")
        for e in errores:
            print(f"   - {e}")
        print("\nCausas posibles:")
        print("  1. Versión de PyTorch diferente (comprueba con la Fase 0)")
        print("  2. Semilla no fijada correctamente (comprueba RANDOM_SEED = 42)")
        print("  3. Datos procesados con una versión diferente del pipeline")
        print("  4. GPU diferente (T4 vs A100 pueden dar diferencias menores en float32)")
